# Executed notebook: 00_setup_uc

Exported from Databricks job run `417115498203055` (task `setup_uc`, task run `21131356802344`).

Result: **SUCCESS** · start 2026-10-05T21:57:23.721000+00:00 · end 2026-10-05T21:58:12.587000+00:00

Run URL: https://fevm-serverless-stable-am1uc2.cloud.databricks.com/?o=7474651880045550#job/1029669706813282/run/417115498203055


# 00 · Unity Catalog setup
Creates the schemas, the Zerobus landing table, synthetic reference data, and least-privilege grants
for the Zerobus producer service principal. All data is synthetic.

In [ ]:
dbutils.widgets.text("catalog", "serverless_stable_am1uc2_catalog")
dbutils.widgets.text("zerobus_sp_app_id", "240a5501-f956-4282-a2a7-ccc7614683a7")
CATALOG = dbutils.widgets.get("catalog")
SP = dbutils.widgets.get("zerobus_sp_app_id")

import os, sys
sys.path.append(os.path.abspath(".."))
from pdm.config import SCHEMA_RAW, SCHEMA_CORE, SCHEMA_ML, SCHEMA_OPS, SENSORS
from pdm.physics import build_station_master

In [ ]:
schemas = {
    SCHEMA_RAW: "Landing zone: Zerobus telemetry and reference data (synthetic).",
    SCHEMA_CORE: "SDP output: cleaned telemetry, window features, live risk scores.",
    SCHEMA_ML: "Training sets and the station failure model.",
    SCHEMA_OPS: "Operational tables (work orders) and governed metric views.",
}
for s, c in schemas.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{s} COMMENT '{c}'")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG} LIKE 'pdm_*'"))

| databaseName |
|---|
| pdm_core |
| pdm_ml |
| pdm_ops |
| pdm_raw |


## Zerobus target table
Zerobus writes into a pre-created **managed Delta** table. It never creates or alters tables.

In [ ]:
sensor_cols = ",\n  ".join(f"{s} DOUBLE" for s in SENSORS)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA_RAW}.sensor_readings (
  event_id STRING COMMENT 'Deterministic id (station + seq); used for de-duplication (Zerobus is at-least-once)',
  gateway_id STRING COMMENT 'Plant edge gateway that pushed the record',
  plant_id STRING,
  line_id STRING,
  station_id STRING,
  ts TIMESTAMP COMMENT 'Sensor sample time (UTC)',
  seq BIGINT COMMENT 'Per-station sequence number',
  {sensor_cols},
  firmware STRING
)
COMMENT 'Raw 1 Hz station telemetry pushed by plant gateways via Lakeflow Connect Zerobus Ingest (synthetic).'
""")
display(spark.sql(f"DESCRIBE TABLE {CATALOG}.{SCHEMA_RAW}.sensor_readings"))

| col_name | data_type | comment |
|---|---|---|
| event_id | string | Deterministic id (station + seq); used for de-duplication (Zerobus is at-least-once) |
| gateway_id | string | Plant edge gateway that pushed the record |
| plant_id | string |  |
| line_id | string |  |
| station_id | string |  |
| ts | timestamp | Sensor sample time (UTC) |
| seq | bigint | Per-station sequence number |
| vibration_rms | double |  |
| bearing_temp_c | double |  |
| motor_current_a | double |  |
| spindle_rpm | double |  |
| hydraulic_pressure_bar | double |  |
| acoustic_db | double |  |
| cycle_time_s | double |  |
| firmware | string |  |


## Reference data: stations and technicians

In [ ]:
stations = spark.createDataFrame(build_station_master())
(stations.write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOG}.{SCHEMA_RAW}.station_master"))
spark.sql(f"COMMENT ON TABLE {CATALOG}.{SCHEMA_RAW}.station_master IS "
          "'Station master data: 3 plants x 4 lines x 8 stations, with nominal sensor operating points (synthetic).'")
display(spark.sql(f"""
SELECT plant_name, station_type, count(*) AS stations
FROM {CATALOG}.{SCHEMA_RAW}.station_master GROUP BY ALL ORDER BY ALL"""))

| plant_name | station_type | stations |
|---|---|---|
| Plant East | cnc_mill | 8 |
| Plant East | conveyor | 4 |
| Plant East | press | 8 |
| Plant East | robot_arm | 4 |
| Plant East | welder | 8 |
| Plant North | cnc_mill | 8 |
| Plant North | conveyor | 4 |
| Plant North | press | 8 |
| Plant North | robot_arm | 4 |
| Plant North | welder | 8 |
| Plant South | cnc_mill | 8 |
| Plant South | conveyor | 4 |
| Plant South | press | 8 |
| Plant South | robot_arm | 4 |
| Plant South | welder | 8 |


In [ ]:
import random
random.seed(11)
first = ["Alex", "Sam", "Jordan", "Taylor", "Morgan", "Casey", "Riley", "Jamie", "Drew", "Avery", "Quinn", "Reese"]
last = ["Novak", "Okafor", "Lindqvist", "Moreau", "Tanaka", "Silva", "Kowalski", "Haddad", "Brennan", "Iyer"]
techs = []
for i, plant in enumerate(["PLT-N", "PLT-S", "PLT-E"] * 6):
    fn, ln = random.choice(first), random.choice(last)
    techs.append(dict(
        technician_id=f"T{i+1:03d}", full_name=f"{fn} {ln}", home_plant_id=plant,
        skill=random.choice(["mechanical", "electrical", "hydraulics"]),
        email=f"{fn.lower()}.{ln.lower()}{i}@example-mfg.test", phone=f"+1-555-01{i:02d}",
    ))
(spark.createDataFrame(techs).write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOG}.{SCHEMA_RAW}.technicians"))
spark.sql(f"COMMENT ON TABLE {CATALOG}.{SCHEMA_RAW}.technicians IS "
          "'Maintenance technicians (synthetic). email/phone are PII and masked for non-supervisors.'")
display(spark.table(f"{CATALOG}.{SCHEMA_RAW}.technicians").limit(5))

| email | full_name | home_plant_id | phone | skill | technician_id |
|---|---|---|---|---|---|
| jamie.brennan0@example-mfg.test | Jamie Brennan | PLT-N | +1-555-0100 | electrical | T001 |
| jamie.brennan1@example-mfg.test | Jamie Brennan | PLT-S | +1-555-0101 | hydraulics | T002 |
| taylor.lindqvist2@example-mfg.test | Taylor Lindqvist | PLT-E | +1-555-0102 | hydraulics | T003 |
| jamie.iyer3@example-mfg.test | Jamie Iyer | PLT-N | +1-555-0103 | mechanical | T004 |
| sam.haddad4@example-mfg.test | Sam Haddad | PLT-S | +1-555-0104 | electrical | T005 |


## Least-privilege grants for the Zerobus producer
Zerobus needs explicit **table-level** `MODIFY` + `SELECT`; schema inheritance is not enough.

In [ ]:
for stmt in [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{SP}`",
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA_RAW} TO `{SP}`",
    f"GRANT MODIFY, SELECT ON TABLE {CATALOG}.{SCHEMA_RAW}.sensor_readings TO `{SP}`",
]:
    spark.sql(stmt)
    print("OK:", stmt)
display(spark.sql(f"SHOW GRANTS `{SP}` ON TABLE {CATALOG}.{SCHEMA_RAW}.sensor_readings"))

OK: GRANT USE CATALOG ON CATALOG serverless_stable_am1uc2_catalog TO `240a5501-f956-4282-a2a7-ccc7614683a7`
OK: GRANT USE SCHEMA ON SCHEMA serverless_stable_am1uc2_catalog.pdm_raw TO `240a5501-f956-4282-a2a7-ccc7614683a7`
OK: GRANT MODIFY, SELECT ON TABLE serverless_stable_am1uc2_catalog.pdm_raw.sensor_readings TO `240a5501-f956-4282-a2a7-ccc7614683a7`



| Principal | ActionType | ObjectType | ObjectKey |
|---|---|---|---|
| 240a5501-f956-4282-a2a7-ccc7614683a7 | MODIFY | TABLE | serverless_stable_am1uc2_catalog.pdm_raw.sensor_readings |
| 240a5501-f956-4282-a2a7-ccc7614683a7 | SELECT | TABLE | serverless_stable_am1uc2_catalog.pdm_raw.sensor_readings |
